In [ ]:
# 현재 노트북 커널에 패키지가 없으면 아래 줄의 주석을 해제해 한 번만 실행
# %pip install pypdfium2 paddleocr

from pathlib import Path
from datetime import datetime
import json
import shutil
import statistics

# torch를 paddle(paddleocr)보다 먼저 import 해야 한다.
# paddlepaddle와 torch가 Windows에서 공유 DLL이 충돌(WinError 127)하는데,
# torch를 먼저 올리면 DLL이 정상 로드된다. (paddleocr 2.10은 내부에서 torch를 끌어온다)
import torch  # noqa: F401

import pypdfium2 as pdfium
from paddleocr import PaddleOCR

# 프로젝트 루트 자동 탐색
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data').exists() and (path / 'data_parsing').exists()
)
DATA_DIR = PROJECT_ROOT / 'data'
DPI = 300
DEVICE = 'cpu'
LOW_CONFIDENCE_THRESHOLD = 0.60
KEEP_RENDERED_IMAGES = False
REPLACE_EXISTING_JSON = False  # False이면 _ocr.json으로 저장

# 실제로 사용한 OCR 엔진 정보. 다음 셀에서 엔진을 만든 뒤 갱신되어 메타데이터에 기록된다.
OCR_META = {
    'extraction_method': 'pypdfium2_300dpi+paddleocr',
    'ocr_engine_api': None,
    'ocr_version': None,
}

TARGETS = [
    {
        'pdf_path': DATA_DIR / '20220406_온라인_식품·건강기능식품·의약외품·화장품_분야_부당광고_사례집-웹게시용(표지수정).pdf',
        'output_dir': PROJECT_ROOT / 'data_parsing' / '해설서, 사례',
        'title': '온라인 식품·건강기능식품·의약외품·화장품 분야 부당광고 사례집',
        'notice_number': None,
        'published_date': '2022-04-06',
    },
    {
        'pdf_path': DATA_DIR / '☆+「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2024-23호，+2024.6.11.).pdf',
        'output_dir': PROJECT_ROOT / 'data_parsing' / 'law',
        'title': '식품등의 부당한 표시 또는 광고의 내용 기준',
        'notice_number': '제2024-23호',
        'published_date': '2024-06-11',
    },
    {
        # HWPX 원본을 PDF로 변환한 최신 개정 고시. (기존 파싱 방식과 형식이 맞지 않아 OCR로 재파싱)
        'pdf_path': DATA_DIR / '★++「식품등의+부당한+표시+또는+광고의+내용+기준」+고시+전문(제2025-79호,+2025.12.4..pdf',
        'output_dir': PROJECT_ROOT / 'data_parsing' / 'law',
        'title': '식품등의 부당한 표시 또는 광고의 내용 기준',
        'notice_number': '제2025-79호',
        'published_date': '2025-12-04',
    },
]

for target in TARGETS:
    if not target['pdf_path'].exists():
        raise FileNotFoundError(target['pdf_path'])
print(f'프로젝트 루트: {PROJECT_ROOT}')
for target in TARGETS:
    print(f"  - {target['pdf_path'].name}  ->  {target['output_dir'].relative_to(PROJECT_ROOT)}")

In [ ]:
def to_list(value):
    if value is None:
        return []
    return value.tolist() if hasattr(value, 'tolist') else list(value)


def normalize_bbox(box):
    values = to_list(box)
    if len(values) == 4 and not isinstance(values[0], (list, tuple)):
        return [float(v) for v in values]
    xs = [float(point[0]) for point in values]
    ys = [float(point[1]) for point in values]
    return [min(xs), min(ys), max(xs), max(ys)]


def render_pdf_pages(pdf_path, image_dir, dpi=300):
    # Poppler 없이 PDFium으로 300 DPI PNG 생성
    image_dir.mkdir(parents=True, exist_ok=True)
    document = pdfium.PdfDocument(str(pdf_path))
    scale = dpi / 72
    image_paths = []

    try:
        for page_index in range(len(document)):
            page = document[page_index]
            bitmap = page.render(scale=scale)
            image = bitmap.to_pil().convert('RGB')
            image_path = image_dir / f'page_{page_index + 1:04d}.png'
            image.save(image_path, format='PNG', optimize=True)
            image_paths.append(image_path)
            bitmap.close()
            page.close()
    finally:
        document.close()

    return image_paths


def extract_v3_blocks(result):
    raw = getattr(result, 'json', result)
    raw = raw() if callable(raw) else raw
    raw = json.loads(raw) if isinstance(raw, str) else raw
    payload = raw.get('res', raw)

    texts = to_list(payload.get('rec_texts'))
    scores = to_list(payload.get('rec_scores'))
    boxes_value = payload.get('rec_boxes')
    if boxes_value is None:
        boxes_value = payload.get('rec_polys')
    boxes = to_list(boxes_value)
    blocks = []

    for text, score, box in zip(texts, scores, boxes):
        text = str(text).strip()
        if not text:
            continue
        blocks.append({
            'text': text,
            'confidence': round(float(score), 6),
            'bbox': normalize_bbox(box),
        })
    return blocks


def extract_v2_blocks(result):
    rows = result[0] if result and len(result) == 1 else result
    blocks = []

    for row in rows or []:
        if not row or len(row) < 2:
            continue
        box, recognition = row[0], row[1]
        text, score = recognition[0], recognition[1]
        text = str(text).strip()
        if not text:
            continue
        blocks.append({
            'text': text,
            'confidence': round(float(score), 6),
            'bbox': normalize_bbox(box),
        })
    return blocks


def ocr_page(ocr_engine, image_path):
    # PaddleOCR 3.x와 2.x 결과 형식을 모두 처리
    if hasattr(ocr_engine, 'predict'):
        results = list(ocr_engine.predict(str(image_path)))
        blocks = []
        for result in results:
            blocks.extend(extract_v3_blocks(result))
    else:
        blocks = extract_v2_blocks(ocr_engine.ocr(str(image_path), cls=True))

    blocks.sort(key=lambda item: (round(item['bbox'][1] / 10), item['bbox'][0]))
    for block in blocks:
        block['low_confidence'] = block['confidence'] < LOW_CONFIDENCE_THRESHOLD

    lines = [block['text'] for block in blocks]
    scores = [block['confidence'] for block in blocks]
    return {
        'text': '\n'.join(lines),
        'lines': lines,
        'character_count': sum(len(line) for line in lines),
        'average_confidence': round(statistics.fmean(scores), 6) if scores else None,
        'low_confidence_block_count': sum(block['low_confidence'] for block in blocks),
        'ocr_blocks': blocks,
    }


def build_output_path(target):
    suffix = '.json' if REPLACE_EXISTING_JSON else '_ocr.json'
    return target['output_dir'] / f"{target['pdf_path'].stem}{suffix}"


def save_json_safely(result, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if REPLACE_EXISTING_JSON and output_path.exists():
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = output_path.with_suffix(f'.backup_{timestamp}.json')
        shutil.copy2(output_path, backup_path)
        print(f'기존 파일 백업: {backup_path}')

    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    temporary_path.replace(output_path)


def parse_pdf_with_ocr(target, ocr_engine):
    pdf_path = target['pdf_path']
    image_dir = PROJECT_ROOT / 'tmp' / 'pdfs' / 'ocr_rendered' / pdf_path.stem
    output_path = build_output_path(target)

    if image_dir.exists():
        shutil.rmtree(image_dir)
    image_paths = render_pdf_pages(pdf_path, image_dir, DPI)

    pages = []
    try:
        for page_number, image_path in enumerate(image_paths, start=1):
            page_result = ocr_page(ocr_engine, image_path)
            pages.append({'page_number': page_number, **page_result})
            print(
                f'[{pdf_path.name}] {page_number}/{len(image_paths)}페이지 '
                f'- {page_result["character_count"]:,}자'
            )
    finally:
        if not KEEP_RENDERED_IMAGES and image_dir.exists():
            shutil.rmtree(image_dir)

    nonempty_pages = sum(bool(page['text'].strip()) for page in pages)
    all_scores = [
        block['confidence']
        for page in pages
        for block in page['ocr_blocks']
    ]
    total_characters = sum(page['character_count'] for page in pages)
    replacement_count = sum(page['text'].count('�') for page in pages)

    result = {
        'document': {
            'title': target['title'],
            'issuing_authority': '식품의약품안전처',
            'notice_number': target['notice_number'],
            'published_date': target['published_date'],
            'source_pdf': str(pdf_path.resolve()),
            'page_count': len(pages),
            'extraction_method': OCR_META['extraction_method'],
            'ocr_language': 'korean',
            'ocr_engine_api': OCR_META['ocr_engine_api'],
            'ocr_version': OCR_META['ocr_version'],
            'dpi': DPI,
            'extracted_at': datetime.now().astimezone().isoformat(),
            'nonempty_page_count': nonempty_pages,
            'nonempty_page_rate': round(nonempty_pages / len(pages), 6) if pages else 0,
            'total_extracted_characters': total_characters,
            'average_ocr_confidence': round(statistics.fmean(all_scores), 6) if all_scores else None,
            'low_confidence_block_count': sum(page['low_confidence_block_count'] for page in pages),
            'replacement_character_count': replacement_count,
        },
        'pages': pages,
    }

    save_json_safely(result, output_path)
    return output_path, result

In [ ]:
# 최초 실행 시 한글 OCR 모델을 자동으로 내려받는다.
# PaddleOCR 3.x(PP-OCRv5, predict API)를 우선 시도하고, 2.x(2.10 등)면 자동으로 내려간다.
try:
    ocr_engine = PaddleOCR(
        lang='korean',
        ocr_version='PP-OCRv5',
        device=DEVICE,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    OCR_META['ocr_engine_api'] = 'paddleocr_3x_predict'
    OCR_META['ocr_version'] = 'PP-OCRv5'
except Exception as exc:
    # PaddleOCR 2.x 호환 (설치본 2.10.0은 PP-OCRv5를 모르므로 AssertionError가 나며, 한글 rec은 PP-OCRv4)
    print(f'PP-OCRv5(3.x) 사용 불가 → PaddleOCR 2.x 경로로 전환: {type(exc).__name__}')
    ocr_engine = PaddleOCR(lang='korean', use_angle_cls=True, use_gpu=(DEVICE != 'cpu'))
    OCR_META['ocr_engine_api'] = 'paddleocr_2x_ocr'
    detected_version = getattr(getattr(ocr_engine, 'args', None), 'ocr_version', None)
    OCR_META['ocr_version'] = detected_version or 'PP-OCRv4'

print(f"OCR 엔진 준비 완료: api={OCR_META['ocr_engine_api']}, version={OCR_META['ocr_version']}")

parsed_results = []
for target in TARGETS[:1]:
    output_path, result = parse_pdf_with_ocr(target, ocr_engine)
    parsed_results.append((output_path, result))
    print(f'저장 완료: {output_path}')

In [ ]:
# 저장 결과 기본 검증
for output_path, result in parsed_results:
    saved = json.loads(output_path.read_text(encoding='utf-8'))
    document = saved['document']
    pages = saved['pages']

    assert document['page_count'] == len(pages)
    assert all(page['page_number'] == index for index, page in enumerate(pages, start=1))
    assert document['total_extracted_characters'] > 0
    assert document['replacement_character_count'] == 0

    print({
        'file': output_path.name,
        'page_count': document['page_count'],
        'nonempty_page_rate': document['nonempty_page_rate'],
        'total_characters': document['total_extracted_characters'],
        'average_confidence': document['average_ocr_confidence'],
        'low_confidence_blocks': document['low_confidence_block_count'],
    })
